# PHASE 4: Feature Engineering — NASA RUL (EDA-Driven)

**Traceability**
- Issue ID: #4 Feature Engineering
- Depends on: EDA findings from `03_eda.ipynb`

## Strategy

Based on EDA analysis, five targeted dataset variants are produced:

| Dataset | Outlier | Features | Scaling | PCA | Use Case |
|---------|---------|----------|---------|-----|----------|
| **DS-A** | None | Raw sensors + clipped RUL | MinMax | No | Baseline |
| **DS-B** | Winsorize 1–99% | Lag+Rolling+Diff+EWMA+HI | MinMax | No | Classical ML (trees, ensembles) |
| **DS-C** | Winsorize 1–99% | Lag+Rolling+Diff+EWMA | MinMax | 95% var | Linear / regularized models |
| **DS-D** | None | Rolling+Diff+HI | MinMax | No | Deep Learning (LSTM, Transformer) |
| **DS-E** | Clip 1–99% + MinMax | Lag+Rolling+Diff+EWMA+HI | Included | No | Full comprehensive |

**Key decisions from EDA:**
- `s_9` and `s_14` outliers (~8%) are failure-relevant — no aggressive clipping for DL.
- Non-linear late-stage acceleration → diff and rolling features are essential.
- PC1 explains 69.6% variance → health index is a strong composite feature.
- 14 correlated sensors → PCA reduction justified for compact models.

**Outputs:** `../data/experiments/DS-{A,B,C,D,E}/train.csv` and `valid.csv`

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore", category=FutureWarning)

# ── Paths ─────────────────────────────────────────────────────────────
DATA_DIR = Path("../data/processed")
EXP_DIR = Path("../data/experiments")
TRAIN_PATH = DATA_DIR / "train_cleaned.csv"
VALID_PATH = DATA_DIR / "valid_cleaned.csv"

# ── Constants ─────────────────────────────────────────────────────────
RUL_CLIP = 125
ROLL_WINDOWS = [5, 10, 20]
LAG_STEPS = [1, 2, 5]
EWMA_SPANS = [5, 10]
RANDOM_STATE = 42


def load_data(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    for col in ["unit_number", "time_cycles"]:
        assert col in df.columns, f"Missing column: {col}"
    assert "RUL" in df.columns, (
        f"Missing RUL column in {path.name}. "
        "RUL must be computed in 02_data_cleaning (train via max_cycle, "
        "valid via RUL_FDxxx.txt ground truth)."
    )
    return df.sort_values(["unit_number", "time_cycles"]).reset_index(drop=True)


df_train_raw = load_data(TRAIN_PATH)
df_valid_raw = load_data(VALID_PATH)

sensor_cols = sorted(
    [c for c in df_train_raw.columns if c.startswith("s_")],
    key=lambda x: int(x.split("_")[1]),
)
setting_cols = [c for c in df_train_raw.columns if c.startswith("setting_")]

print(f"Train: {df_train_raw.shape}  Valid: {df_valid_raw.shape}")
print(f"Sensors ({len(sensor_cols)}): {sensor_cols}")

Train: (20631, 20)  Valid: (13096, 20)
Sensors (14): ['s_2', 's_3', 's_4', 's_7', 's_8', 's_9', 's_11', 's_12', 's_13', 's_14', 's_15', 's_17', 's_20', 's_21']


## 1 — Reusable Pipeline Components

All operations are defined as functions so each dataset variant can be assembled declaratively.

In [2]:
# ── 1a) RUL clipping ──────────────────────────────────────────────────
def clip_rul(df: pd.DataFrame, upper: int = RUL_CLIP) -> pd.DataFrame:
    df = df.copy()
    df["RUL_clipped"] = df["RUL"].clip(upper=upper)
    return df


# ── 1b) Outlier handling ─────────────────────────────────────────────
def winsorize_sensors(
    df_train: pd.DataFrame,
    df_valid: pd.DataFrame,
    sensors: list[str],
    limits: tuple = (0.01, 0.01),
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Winsorize: compute bounds on train, apply to both."""
    df_tr, df_vl = df_train.copy(), df_valid.copy()
    for s in sensors:
        lo = df_tr[s].quantile(limits[0])
        hi = df_tr[s].quantile(1 - limits[1])
        df_tr[s] = df_tr[s].clip(lo, hi)
        df_vl[s] = df_vl[s].clip(lo, hi)
    return df_tr, df_vl


def clip_sensors(
    df_train: pd.DataFrame,
    df_valid: pd.DataFrame,
    sensors: list[str],
    lower_q: float = 0.01,
    upper_q: float = 0.99,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Percentile clip: fit on train, apply to both."""
    df_tr, df_vl = df_train.copy(), df_valid.copy()
    for s in sensors:
        lo = df_tr[s].quantile(lower_q)
        hi = df_tr[s].quantile(upper_q)
        df_tr[s] = df_tr[s].clip(lo, hi)
        df_vl[s] = df_vl[s].clip(lo, hi)
    return df_tr, df_vl


# ── 1c) Scaling ───────────────────────────────────────────────────────
def minmax_scale(
    df_train: pd.DataFrame,
    df_valid: pd.DataFrame,
    cols: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame, MinMaxScaler]:
    df_tr, df_vl = df_train.copy(), df_valid.copy()
    scaler = MinMaxScaler()
    df_tr[cols] = scaler.fit_transform(df_tr[cols])
    df_vl[cols] = scaler.transform(df_vl[cols])
    return df_tr, df_vl, scaler


# ── 1d) Feature engineering ──────────────────────────────────────────
def add_lag_features(df: pd.DataFrame, sensors: list[str]) -> pd.DataFrame:
    out = df.copy()
    grouped = out.groupby("unit_number", group_keys=False)
    for s in sensors:
        for lag in LAG_STEPS:
            out[f"{s}_lag_{lag}"] = grouped[s].shift(lag)
    return out


def add_rolling_features(df: pd.DataFrame, sensors: list[str]) -> pd.DataFrame:
    out = df.copy()
    grouped = out.groupby("unit_number", group_keys=False)
    for s in sensors:
        for w in ROLL_WINDOWS:
            out[f"{s}_rmean_{w}"] = grouped[s].transform(
                lambda x: x.rolling(w, min_periods=1).mean()
            )
            out[f"{s}_rstd_{w}"] = grouped[s].transform(
                lambda x: x.rolling(w, min_periods=1).std().fillna(0)
            )
    return out


def add_diff_features(df: pd.DataFrame, sensors: list[str]) -> pd.DataFrame:
    out = df.copy()
    grouped = out.groupby("unit_number", group_keys=False)
    for s in sensors:
        out[f"{s}_diff1"] = grouped[s].diff().fillna(0)
    return out


def add_ewma_features(df: pd.DataFrame, sensors: list[str]) -> pd.DataFrame:
    out = df.copy()
    grouped = out.groupby("unit_number", group_keys=False)
    for s in sensors:
        for span in EWMA_SPANS:
            out[f"{s}_ewma_{span}"] = grouped[s].transform(
                lambda x, sp=span: x.ewm(span=sp, min_periods=1).mean()
            )
    return out


def add_health_index(
    df_train: pd.DataFrame,
    df_valid: pd.DataFrame,
    sensors: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """PC1-based health index: higher = healthier. Fit on train only."""
    df_tr, df_vl = df_train.copy(), df_valid.copy()
    pca = PCA(n_components=1, random_state=RANDOM_STATE)

    hi_train = pca.fit_transform(df_tr[sensors]).ravel()
    hi_valid = pca.transform(df_vl[sensors]).ravel()

    hi_scaler = MinMaxScaler()
    hi_all = np.concatenate([hi_train, hi_valid]).reshape(-1, 1)
    hi_all = hi_scaler.fit_transform(hi_all).ravel()

    # Invert so high RUL → high health
    df_tr["health_index"] = 1.0 - hi_all[: len(df_tr)]
    df_vl["health_index"] = 1.0 - hi_all[len(df_tr) :]
    return df_tr, df_vl


# ── 1e) PCA dimensionality reduction ─────────────────────────────────
def apply_pca(
    df_train: pd.DataFrame,
    df_valid: pd.DataFrame,
    feature_cols: list[str],
    variance_threshold: float = 0.95,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    """Replace feature_cols with PCA components retaining given variance."""
    pca = PCA(n_components=variance_threshold, random_state=RANDOM_STATE)

    tr_pca = pca.fit_transform(df_train[feature_cols])
    vl_pca = pca.transform(df_valid[feature_cols])

    n_comp = tr_pca.shape[1]
    pc_names = [f"PC_{i+1}" for i in range(n_comp)]

    df_tr = df_train.drop(columns=feature_cols).copy()
    df_vl = df_valid.drop(columns=feature_cols).copy()

    for i, name in enumerate(pc_names):
        df_tr[name] = tr_pca[:, i]
        df_vl[name] = vl_pca[:, i]

    print(f"  PCA: {len(feature_cols)} features → {n_comp} components ({variance_threshold*100:.0f}% variance)")
    return df_tr, df_vl, pc_names


# ── 1f) Cleanup and export ───────────────────────────────────────────
def finalize_and_save(
    df_train: pd.DataFrame,
    df_valid: pd.DataFrame,
    dataset_name: str,
    drop_meta: bool = True,
) -> None:
    """Drop NaN rows from lag/rolling, validate, and save."""
    # Fill NaN from lag features (first rows per engine)
    df_train = df_train.fillna(0)
    df_valid = df_valid.fillna(0)

    out_dir = EXP_DIR / dataset_name
    out_dir.mkdir(parents=True, exist_ok=True)

    # Drop columns not needed for modeling
    meta_cols = ["unit_number", "time_cycles", "RUL"] + setting_cols
    if drop_meta:
        keep_tr = [c for c in df_train.columns if c not in meta_cols]
        keep_vl = [c for c in df_valid.columns if c not in meta_cols]
    else:
        keep_tr = list(df_train.columns)
        keep_vl = list(df_valid.columns)

    df_train[keep_tr].to_csv(out_dir / "train.csv", index=False)
    df_valid[keep_vl].to_csv(out_dir / "valid.csv", index=False)

    tr_nan = int(df_train[keep_tr].isna().sum().sum())
    vl_nan = int(df_valid[keep_vl].isna().sum().sum())

    print(f"  ✅ {dataset_name}: train {df_train[keep_tr].shape}, valid {df_valid[keep_vl].shape} | NaN: train={tr_nan}, valid={vl_nan}")
    assert "RUL_clipped" in keep_tr, f"{dataset_name}: RUL_clipped missing from output"

print("✅ Pipeline components defined")

✅ Pipeline components defined


## 2 — DS-A: Baseline (raw sensors, MinMax scaled, clipped RUL)

**Rationale:** Minimal processing. Establishes the floor performance that all other variants must beat. No outlier treatment, no engineered features — just scaled sensors and piecewise RUL.

In [3]:
print("═" * 60)
print("DS-A: Baseline")
print("═" * 60)

tr_a = clip_rul(df_train_raw)
vl_a = clip_rul(df_valid_raw)

tr_a, vl_a, _ = minmax_scale(tr_a, vl_a, sensor_cols)

finalize_and_save(tr_a, vl_a, "DS-A")

════════════════════════════════════════════════════════════
DS-A: Baseline
════════════════════════════════════════════════════════════
  ✅ DS-A: train (20631, 15), valid (13096, 15) | NaN: train=0, valid=0


## 3 — DS-B: Classical ML (Winsorize + Full features + Health Index)

**Rationale:** The EDA showed non-linear late-stage trends, so lag, rolling, diff and EWMA features capture rate-of-change and local trajectory shape. Winsorizing at 1–99% compresses the `s_9`/`s_14` tails gently without destroying failure-relevant signal. Health index adds a powerful PCA-derived composite. Designed for Random Forest, XGBoost, and similar tree/ensemble models.

In [4]:
print("═" * 60)
print("DS-B: Classical ML (Winsorize + Full features + HI)")
print("═" * 60)

tr_b = clip_rul(df_train_raw)
vl_b = clip_rul(df_valid_raw)

# Outlier: winsorize
tr_b, vl_b = winsorize_sensors(tr_b, vl_b, sensor_cols)

# Scale sensors
tr_b, vl_b, _ = minmax_scale(tr_b, vl_b, sensor_cols)

# Health index (on scaled sensors)
tr_b, vl_b = add_health_index(tr_b, vl_b, sensor_cols)

# Feature engineering
tr_b = add_lag_features(tr_b, sensor_cols)
tr_b = add_rolling_features(tr_b, sensor_cols)
tr_b = add_diff_features(tr_b, sensor_cols)
tr_b = add_ewma_features(tr_b, sensor_cols)

vl_b = add_lag_features(vl_b, sensor_cols)
vl_b = add_rolling_features(vl_b, sensor_cols)
vl_b = add_diff_features(vl_b, sensor_cols)
vl_b = add_ewma_features(vl_b, sensor_cols)

finalize_and_save(tr_b, vl_b, "DS-B")

════════════════════════════════════════════════════════════
DS-B: Classical ML (Winsorize + Full features + HI)
════════════════════════════════════════════════════════════
  ✅ DS-B: train (20631, 184), valid (13096, 184) | NaN: train=0, valid=0


## 4 — DS-C: Compact ML (Winsorize + Full features + PCA 95%)

**Rationale:** Same feature engineering as DS-B, but the high dimensionality (~300+ features) is compressed via PCA retaining 95% variance. The correlation heatmap showed strong redundancy, and PCA at 95% preserves signal while eliminating collinearity. Designed for linear models, SVR, and regularized regression.

In [5]:
print("═" * 60)
print("DS-C: Compact ML (Winsorize + Full features + PCA 95%)")
print("═" * 60)

tr_c = clip_rul(df_train_raw)
vl_c = clip_rul(df_valid_raw)

# Outlier: winsorize
tr_c, vl_c = winsorize_sensors(tr_c, vl_c, sensor_cols)

# Scale sensors
tr_c, vl_c, _ = minmax_scale(tr_c, vl_c, sensor_cols)

# Feature engineering (same as DS-B but no health_index — PCA replaces it)
tr_c = add_lag_features(tr_c, sensor_cols)
tr_c = add_rolling_features(tr_c, sensor_cols)
tr_c = add_diff_features(tr_c, sensor_cols)
tr_c = add_ewma_features(tr_c, sensor_cols)

vl_c = add_lag_features(vl_c, sensor_cols)
vl_c = add_rolling_features(vl_c, sensor_cols)
vl_c = add_diff_features(vl_c, sensor_cols)
vl_c = add_ewma_features(vl_c, sensor_cols)

# Fill NaN before PCA (lag-created NaNs)
tr_c = tr_c.fillna(0)
vl_c = vl_c.fillna(0)

# PCA on all engineered features (excluding meta and target)
meta_cols_pca = {"unit_number", "time_cycles", "RUL", "RUL_clipped"} | set(setting_cols)
feat_cols_c = [c for c in tr_c.columns if c not in meta_cols_pca]

tr_c, vl_c, pc_names = apply_pca(tr_c, vl_c, feat_cols_c, variance_threshold=0.95)

finalize_and_save(tr_c, vl_c, "DS-C")

════════════════════════════════════════════════════════════
DS-C: Compact ML (Winsorize + Full features + PCA 95%)
════════════════════════════════════════════════════════════
  PCA: 182 features → 33 components (95% variance)
  ✅ DS-C: train (20631, 34), valid (13096, 34) | NaN: train=0, valid=0


## 5 — DS-D: Deep Learning (Raw sensors + Rolling + Diff + Health Index)

**Rationale:** Sequence models (LSTM, Transformer) learn temporal patterns internally, so heavy feature engineering adds noise rather than signal. We keep: (1) raw scaled sensors to preserve fine-grained interactions, (2) rolling mean/std to provide local context, (3) diff to highlight rate-of-change, and (4) health index as a single degradation summary. No outlier clipping — the EDA showed `s_9`/`s_14` extremes are failure-relevant and DL models handle them well.

In [6]:
print("═" * 60)
print("DS-D: Deep Learning (Raw + Rolling + Diff + HI)")
print("═" * 60)

tr_d = clip_rul(df_train_raw)
vl_d = clip_rul(df_valid_raw)

# No outlier treatment — preserve extremes for sequence models
# Scale sensors
tr_d, vl_d, _ = minmax_scale(tr_d, vl_d, sensor_cols)

# Health index
tr_d, vl_d = add_health_index(tr_d, vl_d, sensor_cols)

# Lighter features: rolling + diff only (no lag, no EWMA — model learns temporal)
tr_d = add_rolling_features(tr_d, sensor_cols)
tr_d = add_diff_features(tr_d, sensor_cols)

vl_d = add_rolling_features(vl_d, sensor_cols)
vl_d = add_diff_features(vl_d, sensor_cols)

finalize_and_save(tr_d, vl_d, "DS-D")

════════════════════════════════════════════════════════════
DS-D: Deep Learning (Raw + Rolling + Diff + HI)
════════════════════════════════════════════════════════════
  ✅ DS-D: train (20631, 114), valid (13096, 114) | NaN: train=0, valid=0


## 6 — DS-E: Full Comprehensive (Clip + MinMax + All features + Health Index)

**Rationale:** Maximum feature coverage with moderate outlier compression. Percentile clipping at 1–99% is slightly more aggressive than winsorizing (hard boundary vs. soft compression) — paired with MinMax scaling, this produces well-conditioned inputs. All feature types included plus health index. Designed for stacking ensembles and ablation comparison against DS-B.

In [7]:
print("═" * 60)
print("DS-E: Full Comprehensive (Clip 1-99% + All features + HI)")
print("═" * 60)

tr_e = clip_rul(df_train_raw)
vl_e = clip_rul(df_valid_raw)

# Outlier: percentile clip
tr_e, vl_e = clip_sensors(tr_e, vl_e, sensor_cols)

# Scale sensors
tr_e, vl_e, _ = minmax_scale(tr_e, vl_e, sensor_cols)

# Health index
tr_e, vl_e = add_health_index(tr_e, vl_e, sensor_cols)

# All feature types
tr_e = add_lag_features(tr_e, sensor_cols)
tr_e = add_rolling_features(tr_e, sensor_cols)
tr_e = add_diff_features(tr_e, sensor_cols)
tr_e = add_ewma_features(tr_e, sensor_cols)

vl_e = add_lag_features(vl_e, sensor_cols)
vl_e = add_rolling_features(vl_e, sensor_cols)
vl_e = add_diff_features(vl_e, sensor_cols)
vl_e = add_ewma_features(vl_e, sensor_cols)

finalize_and_save(tr_e, vl_e, "DS-E")

════════════════════════════════════════════════════════════
DS-E: Full Comprehensive (Clip 1-99% + All features + HI)
════════════════════════════════════════════════════════════
  ✅ DS-E: train (20631, 184), valid (13096, 184) | NaN: train=0, valid=0


## 7 — Dataset Summary and Validation

In [8]:
print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)

summary_rows = []
for ds_name in ["DS-A", "DS-B", "DS-C", "DS-D", "DS-E"]:
    ds_dir = EXP_DIR / ds_name
    tr = pd.read_csv(ds_dir / "train.csv")
    vl = pd.read_csv(ds_dir / "valid.csv")
    summary_rows.append({
        "Dataset": ds_name,
        "Train Rows": tr.shape[0],
        "Valid Rows": vl.shape[0],
        "Features": tr.shape[1] - 1,  # exclude RUL_clipped
        "Train NaN": int(tr.isna().sum().sum()),
        "Valid NaN": int(vl.isna().sum().sum()),
        "Has RUL_clipped": "RUL_clipped" in tr.columns,
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))
print()

# Show feature column names for each dataset
for ds_name in ["DS-A", "DS-B", "DS-C", "DS-D", "DS-E"]:
    ds_dir = EXP_DIR / ds_name
    tr = pd.read_csv(ds_dir / "train.csv", nrows=1)
    feat_cols = [c for c in tr.columns if c != "RUL_clipped"]
    print(f"\n{ds_name} features ({len(feat_cols)}):")
    print(f"  {feat_cols[:10]}{'...' if len(feat_cols) > 10 else ''}")

print("\n✅ All datasets validated and ready for modeling")

DATASET SUMMARY
Dataset  Train Rows  Valid Rows  Features  Train NaN  Valid NaN  Has RUL_clipped
   DS-A       20631       13096        14          0          0             True
   DS-B       20631       13096       183          0          0             True
   DS-C       20631       13096        33          0          0             True
   DS-D       20631       13096       113          0          0             True
   DS-E       20631       13096       183          0          0             True


DS-A features (14):
  ['s_2', 's_3', 's_4', 's_7', 's_8', 's_9', 's_11', 's_12', 's_13', 's_14']...

DS-B features (183):
  ['s_2', 's_3', 's_4', 's_7', 's_8', 's_9', 's_11', 's_12', 's_13', 's_14']...

DS-C features (33):
  ['PC_1', 'PC_2', 'PC_3', 'PC_4', 'PC_5', 'PC_6', 'PC_7', 'PC_8', 'PC_9', 'PC_10']...

DS-D features (113):
  ['s_2', 's_3', 's_4', 's_7', 's_8', 's_9', 's_11', 's_12', 's_13', 's_14']...

DS-E features (183):
  ['s_2', 's_3', 's_4', 's_7', 's_8', 's_9', 's_11', 's_12', '